# 📦 Mercari Price Analysis — 01: Data Loading & Initial Inspection

> **Dataset**: [Mercari Price Suggestion Challenge](https://www.kaggle.com/c/mercari-price-suggestion-challenge) (Kaggle)  
> **Sample**: 50,000 rows · 8 columns  
> **Goal**: Load the raw dataset and profile its structure, data types, and quality issues before any cleaning or feature engineering.

---

## Imports

In [ ]:
import numpy as np # for numerical compute
import pandas as pd # for data manipulation and analysis

## Load The Dataset

In [4]:
df = pd.read_csv("../data/mercari_sample.csv")

## Initial Inspection

A structured first look at the dataset: dimensions, column types, null counts, sample rows, and summary statistics.  
The goal is to surface data quality problems *before* investing time in analysis.

In [5]:
# (50,000 rows × 8 columns ≈ 3.1 MB — safe to hold fully in RAM)
df.shape

(50000, 8)

In [6]:
# Immediately shows which columns contain nulls and need cleaning
df.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 50000 entries, 0 to 49999
Data columns (total 8 columns):
 #   Column             Non-Null Count  Dtype  
---  ------             --------------  -----  
 0   train_id           50000 non-null  int64  
 1   name               50000 non-null  object 
 2   item_condition_id  50000 non-null  int64  
 3   category_name      49763 non-null  object 
 4   brand_name         28416 non-null  object 
 5   price              50000 non-null  float64
 6   shipping           50000 non-null  int64  
 7   item_description   50000 non-null  object 
dtypes: float64(1), int64(3), object(4)
memory usage: 3.1+ MB


In [7]:
# Sanity-check that columns parsed correctly and values look as expected
df.head()

,train_id,name,item_condition_id,category_name,brand_name,price,shipping,item_description
0,0,MLB Cincinnati Reds T Shirt Size XL,3,Men/Tops/T-shirts,NaN,10.0,1,No description yet
1,1,Razer BlackWidow Chroma Keyboard,3,Electronics/Computers & Tablets/Components & P...,Razer,52.0,0,This keyboard is in great condition and works ...
2,2,AVA-VIV Blouse,1,Women/Tops & Blouses/Blouse,Target,10.0,1,Adorable top with a hint of lace and a key hol...
3,3,Leather Horse Statues,1,Home/Home Décor/Home Décor Accents,NaN,35.0,1,New with tags. Leather horses. Retail for [rm]...
4,4,24K GOLD plated rose,1,Women/Jewelry/Necklaces,NaN,44.0,0,Complete with certificate of authenticity


In [8]:
# Key signals: price min=0 (free items exist), price max=1,506 (heavy right skew / outliers present)
df.describe().drop(columns=["train_id"])

,item_condition_id,price,shipping
count,50000.000000,50000.000000,50000.000000
mean,1.908660,26.651450,0.448340
std,0.904706,38.208462,0.497329
min,1.000000,0.000000,0.000000
25%,1.000000,10.000000,0.000000
50%,2.000000,17.000000,0.000000
75%,3.000000,29.000000,1.000000
max,5.000000,1506.000000,1.000000


In [9]:
# Null rate per column — drives the imputation strategy:
#   < 5%  →  rows can be dropped without significant data loss
#   ≥ 5%  →  fill with a sentinel value (e.g. "Unknown") to preserve row count
(
    df.isnull().sum() / len(df) * 100
).round(3).rename("null_%")

train_id              0.000
name                  0.000
item_condition_id     0.000
category_name         0.474
brand_name           43.168
price                 0.000
shipping              0.000
item_description      0.000
Name: null_%, dtype: float64

In [10]:
# Confirm there are no exact duplicate rows
df.duplicated().sum()

np.int64(0)